In [9]:
class environment:
    def __init__(self, status):
        self.status = status
        self.agent_location = "A"

    def percept(self):
        return (self.agent_location, self.status[self.agent_location])

    def execute_action(self, action):
        if action == "Suck":
            self.status[self.agent_location] = "Clean"
        elif action == "Right":
            self.agent_location = "B"  # Fixed: Update agent location
        elif action == "Left":
            self.agent_location = "A"  # Fixed: Update agent location


class simpleReflexagent:
    def act(self, percept):
        location, status = percept
        if status == "Dirty":
            return "Suck"              # Fixed: Return the action "Suck"
        elif location == "A":
            return "Right"
        elif location == "B":
            return "Left"

env = environment({"A": "Dirty", "B": "Dirty"}) 
agent = simpleReflexagent() 
  
for step in range(5): 
    percept = env.percept() 
    action = agent.act(percept) 
    print(f"Step {step}: Percept={percept}, Action={action}") 
    env.execute_action(action)

Step 0: Percept=('A', 'Dirty'), Action=Suck
Step 1: Percept=('A', 'Clean'), Action=Right
Step 2: Percept=('B', 'Dirty'), Action=Suck
Step 3: Percept=('B', 'Clean'), Action=Left
Step 4: Percept=('A', 'Clean'), Action=Right


In [12]:
class modelbasedreflexagent:
    def __init__(self):
        self.model = {"A": None, "B": None}

    def update_state(self, percept):
        location, status = percept
        self.model[location] = status

    def act(self, percept):
        location, status = percept
        self.update_state(percept)

        if self.model["A"] == "Clean" and self.model["B"] == "Clean":
            return "Stop"
        elif status == "Dirty":
            return "Suck"        # Fixed: Return "Suck" action
        elif location == "A":
            return "Right"
        else:
            return "Left"        # Fixed: Return "Left" action


env = environment({"A": "Dirty", "B": "Dirty"})
agent = modelbasedreflexagent()

for step in range(6):
    percept = env.percept()
    action = agent.act(percept)
    print(f"Step {step}: Percept={percept}, Model={agent.model}, Action={action}") 
    if action == "Stop":
        break
    env.execute_action(action)

Step 0: Percept=('A', 'Dirty'), Model={'A': 'Dirty', 'B': None}, Action=Suck
Step 1: Percept=('A', 'Clean'), Model={'A': 'Clean', 'B': None}, Action=Right
Step 2: Percept=('B', 'Dirty'), Model={'A': 'Clean', 'B': 'Dirty'}, Action=Suck
Step 3: Percept=('B', 'Clean'), Model={'A': 'Clean', 'B': 'Clean'}, Action=Stop


In [14]:
import random

class RoomEnvironment:
    def __init__(self, initial_temp=18, window_open=False):
        self.temperature = initial_temp
        self.window_open = window_open

    def percept(self):
        return (self.temperature, self.window_open)

    def step_environment(self, action):
        if action == "Heat":
            self.temperature += 1.5
        elif action == "Cool":
            self.temperature -= 1.5

        self.temperature += random.uniform(-0.5, 0.5)

        if random.random() < 0.2:
            self.window_open = not self.window_open

class SimpleReflexClimateAgent:
    def act(self, percept):
        temperature, window_open = percept

        if window_open:
            return "Idle"
        elif temperature < 20:
            return "Heat"
        elif temperature > 24:
            return "Cool"
        else:
            return "Idle"

class ModelBasedClimateAgent:
    def __init__(self):
        self.model = {"temperature": None, "window_open": None}
        self.consecutive_heat_or_cool = 0

    def update_state(self, percept):
        temperature, window_open = percept
        self.model["temperature"] = temperature
        self.model["window_open"] = window_open

    def act(self, percept):
        self.update_state(percept)
        temperature, window_open = percept

        # Determine target action
        if window_open:
            action = "Idle"
        elif temperature < 20:
            action = "Heat"
        elif temperature > 24:
            action = "Cool"
        else:
            action = "Idle"

        # Update tracking state
        if action in ("Heat", "Cool"):
            self.consecutive_heat_or_cool += 1
        else:
            self.consecutive_heat_or_cool = 0

        # Check maintenance condition
        alert = None
        if self.consecutive_heat_or_cool > 3:
            alert = "Maintenance Alert: unit running too long"

        return action, alert

# --- Simulation: Simple Reflex Agent --- 
print("=== Simple Reflex Climate Agent ===") 
random.seed(1) 
env = RoomEnvironment(initial_temp=17, window_open=False) 
simple_agent = SimpleReflexClimateAgent() 

for step in range(8): 
    percept = env.percept() 
    action = simple_agent.act(percept) 
    print(f"Step {step}: Temp={percept[0]:.1f}, Window Open={percept[1]}, Action={action}") 
    env.step_environment(action) 

# --- Simulation: Model-Based Agent --- 
print("\n=== Model-Based Climate Agent ===") 
random.seed(1) 
env = RoomEnvironment(initial_temp=17, window_open=False) 
model_agent = ModelBasedClimateAgent() 

for step in range(8): 
    percept = env.percept() 
    action, alert = model_agent.act(percept) 
    print(f"Step {step}: Temp={percept[0]:.1f}, Window Open={percept[1]}, " 
          f"Model={model_agent.model}, Action={action}" 
          + (f" -> {alert}" if alert else "")) 
    env.step_environment(action)

=== Simple Reflex Climate Agent ===
Step 0: Temp=17.0, Window Open=False, Action=Heat
Step 1: Temp=18.1, Window Open=False, Action=Heat
Step 2: Temp=19.9, Window Open=False, Action=Heat
Step 3: Temp=21.4, Window Open=False, Action=Idle
Step 4: Temp=21.5, Window Open=False, Action=Idle
Step 5: Temp=21.1, Window Open=True, Action=Idle
Step 6: Temp=21.5, Window Open=True, Action=Idle
Step 7: Temp=21.7, Window Open=False, Action=Idle

=== Model-Based Climate Agent ===
Step 0: Temp=17.0, Window Open=False, Model={'temperature': 17, 'window_open': False}, Action=Heat
Step 1: Temp=18.1, Window Open=False, Model={'temperature': 18.1343642441124, 'window_open': False}, Action=Heat
Step 2: Temp=19.9, Window Open=False, Model={'temperature': 19.898138863089017, 'window_open': False}, Action=Heat
Step 3: Temp=21.4, Window Open=False, Model={'temperature': 21.393573950180958, 'window_open': False}, Action=Idle
Step 4: Temp=21.5, Window Open=False, Model={'temperature': 21.54516692290372, 'window_op

In [18]:
class hatchenvironment:
    def __init__(self, hatch_level=20, hatch_open=False):
        self.hatch_level = hatch_level
        self.hatch_open = hatch_open

    def percept(self):
        return self.hatch_level, self.hatch_open

    def execute_action(self, action):
        if action == "Fill":
            self.hatch_level += 8

        elif action == "Drain":
            self.hatch_level -= 8

        # Household consumption happens after every action.
        self.hatch_level -= 3

        # Keep the water level between 0 and 100.
        self.hatch_level = max(0, min(100, self.hatch_level))


class sinmplereflexhatchagent:
    def act(self, percept):
        hatch_level, hatch_open = percept

        if hatch_open:
            return "Idle"

        elif hatch_level < 40:
            return "Fill"

        elif hatch_level > 60:
            return "Drain"

        else:
            return "Idle"


class modelbasedclimateagnet:
    def __init__(self):
        self.model = {
            "hatch_level": None,
            "hatch_open": None
        }

        self.consecutive_fill_drain = 0
        self.previous_action = None

    # These methods are outside __init__, but inside the class.
    def update_state(self, percept):
        hatch_level, hatch_open = percept

        self.model["hatch_level"] = hatch_level
        self.model["hatch_open"] = hatch_open

    def act(self, percept):
        self.update_state(percept)
        hatch_level, hatch_open = percept

        # Select an action without returning yet.
        if hatch_open:
            action = "Idle"

        elif hatch_level < 40:
            action = "Fill"

        elif hatch_level > 60:
            action = "Drain"

        else:
            action = "Idle"

        # Track consecutive operation of the same pump.
        if action == "Idle":
            self.consecutive_fill_drain = 0

        elif action == self.previous_action:
            self.consecutive_fill_drain += 1

        else:
            self.consecutive_fill_drain = 1

        # Remember this action for the next call.
        self.previous_action = action

        alert = None

        if self.consecutive_fill_drain > 3:
            alert = "Maintenance Alert"

        return action, alert

In [19]:
print("SIMPLE REFLEX AGENT")

env = hatchenvironment()
simple_agent = sinmplereflexhatchagent()

for step in range(1, 13):
    # Set the hatch state before sensing.
    env.hatch_open = step in (6, 7)

    percept = env.percept()
    action = simple_agent.act(percept)

    env.execute_action(action)

    print(
        f"Step {step}: "
        f"Start={percept[0]}, "
        f"Hatch Open={percept[1]}, "
        f"Action={action}, "
        f"End={env.hatch_level}"
    )


print("\nMODEL-BASED AGENT")

env = hatchenvironment()
model_agent = modelbasedclimateagnet()

for step in range(1, 13):
    env.hatch_open = step in (6, 7)

    percept = env.percept()

    # The model-based agent returns two values.
    action, alert = model_agent.act(percept)

    env.execute_action(action)

    print(
        f"Step {step}: "
        f"Start={percept[0]}, "
        f"Hatch Open={percept[1]}, "
        f"Action={action}, "
        f"End={env.hatch_level}, "
        f"Count={model_agent.consecutive_fill_drain}, "
        f"Alert={alert}"
    )

SIMPLE REFLEX AGENT
Step 1: Start=20, Hatch Open=False, Action=Fill, End=25
Step 2: Start=25, Hatch Open=False, Action=Fill, End=30
Step 3: Start=30, Hatch Open=False, Action=Fill, End=35
Step 4: Start=35, Hatch Open=False, Action=Fill, End=40
Step 5: Start=40, Hatch Open=False, Action=Idle, End=37
Step 6: Start=37, Hatch Open=True, Action=Idle, End=34
Step 7: Start=34, Hatch Open=True, Action=Idle, End=31
Step 8: Start=31, Hatch Open=False, Action=Fill, End=36
Step 9: Start=36, Hatch Open=False, Action=Fill, End=41
Step 10: Start=41, Hatch Open=False, Action=Idle, End=38
Step 11: Start=38, Hatch Open=False, Action=Fill, End=43
Step 12: Start=43, Hatch Open=False, Action=Idle, End=40

MODEL-BASED AGENT
Step 1: Start=20, Hatch Open=False, Action=Fill, End=25, Count=1, Alert=None
Step 2: Start=25, Hatch Open=False, Action=Fill, End=30, Count=2, Alert=None
Step 3: Start=30, Hatch Open=False, Action=Fill, End=35, Count=3, Alert=None
Step 4: Start=35, Hatch Open=False, Action=Fill, End=40, 

In [22]:
from collections import deque
 
def bfs(graph, start, goal):
    frontier = deque([[start]])    # each frontier item is a PATH (list of states)
    explored = set()
 
    while frontier:
        path = frontier.popleft()      # FIFO: remove from the FRONT
        current = path[-1]
 
        if current == goal:
            return path                # solution found
 
        if current not in explored:
            explored.add(current)
            for neighbor in graph.get(current, []):
                if neighbor not in explored:
                    frontier.append(path + [neighbor])   # add to the BACK
 
    return None   # no solution found


In [23]:
graph = {
    "S": ["A", "B"],
    "A": ["S", "C"],
    "B": ["S", "D"],
    "C": ["A", "G"],
    "D": ["B", "G"],
    "G": ["C", "D"]
}
 
result = bfs(graph, "S", "G")
print("Path found:", result)


Path found: ['S', 'A', 'C', 'G']


In [28]:
from collections import deque

def bfs_me(current , start , goal):
    frontier = deque([[start]])
    explored = set()

    while frontier:
        path = frontier.popleft()
        current = path[-1]

        if current == goal:
            return path

        if current not in explored:
            explored.add(current)
            for neighbour in graph.get(current , []):
                if neighbour not in explored:
                    frontier.append(path + [neighbour])

    return None



In [29]:
graph = {
    "S": ["A", "B"],
    "A": ["S", "C"],
    "B": ["S", "D"],
    "C": ["A", "G"],
    "D": ["B", "G"],
    "G": ["C", "D"]
}
 
result = bfs_me(graph, "S", "G")
print("Path found:", result)


Path found: ['S', 'A', 'C', 'G']


In [30]:
def dfs(current , start , goal):
    frontier = [[start]]
    explored = set()

    while frontier:
        path = frontier.pop()
        current = path[-1]

        if current == goal:
            return path

        if current not in explored:
            explored.add(current)
            for neighour in graph.get(current , []):
                if neighour not in explored:
                    frontier.append(path + [neighour])
    return None

In [31]:
result = dfs(graph, "S", "G")
print("Path found:", result)


Path found: ['S', 'B', 'D', 'G']


In [41]:
def bfs_with_trace(current , start , goal):
    frontier = deque([[start]])
    explored = set()
    trace = []

    while frontier:
        path = frontier.popleft()
        current = path[-1]

        if current not in explored:
            explored.add(current)
            for neighbour in graph.get(current , []):
                if neighbour not in explored:
                    frontier.append(path + [neighbour])

        frontier_stats = [p[-1] for p in frontier]
        trace.append({"current" : current , "frontier" : frontier_stats , "explored" : set(explored)})

        if current == goal:
            return path , trace

    return None , trace


In [42]:
result = bfs_with_trace(graph, "S", "G")
print("Path found:", result)


Path found: (['S', 'A', 'C', 'G'], [{'current': 'S', 'frontier': ['A', 'B'], 'explored': {'S'}}, {'current': 'A', 'frontier': ['B', 'C'], 'explored': {'S', 'A'}}, {'current': 'B', 'frontier': ['C', 'D'], 'explored': {'S', 'A', 'B'}}, {'current': 'C', 'frontier': ['D', 'G'], 'explored': {'C', 'S', 'A', 'B'}}, {'current': 'D', 'frontier': ['G', 'G'], 'explored': {'S', 'D', 'A', 'C', 'B'}}, {'current': 'G', 'frontier': ['G'], 'explored': {'S', 'D', 'A', 'G', 'C', 'B'}}])


In [45]:
import numpy as np

from collections import deque

maze = np.array([
    [0, 0, 1, 0, 0],
    [0, 1, 1, 0, 0],
    [0, 0, 0, 0, 1],
    [1, 1, 0, 1, 0],
    [0, 0, 0, 0, 0]
])

start = (0,0)
goal = (4,4)

def get_neigbours(state , maze):
    row , col = state
    candidates = [(row-1,col) , (row+1,col) , (row,col-1) , (row,col+1)]
    valid_neighboure = []

    for r,c in candidates:
        if 0<=r<maze.shape[0] and 0<=c<maze.shape[1]:
            if(maze[r,c] == 0):
                valid_neighboure.append((r,c))

    return valid_neighboure

def bfs_maze(maze, start, goal):
    frontier = deque([[start]])
    explored = set()
    nodes_explored = 0
 
    while frontier:
        path = frontier.popleft()
        current = path[-1]
 
        if current == goal:
            return path, nodes_explored
 
        if current not in explored:
            explored.add(current)
            nodes_explored += 1
            for neighbor in get_neigbours(current, maze):
                if neighbor not in explored:
                    frontier.append(path + [neighbor])
 
    return None, nodes_explored

def dfs_maze(maze, start, goal):
    frontier = [[start]]
    explored = set()
    nodes_explored = 0
 
    while frontier:
        path = frontier.pop()
        current = path[-1]
 
        if current == goal:
            return path, nodes_explored
 
        if current not in explored:
            explored.add(current)
            nodes_explored += 1
            for neighbor in get_neigbours(current, maze):
                if neighbor not in explored:
                    frontier.append(path + [neighbor])
 
    return None, nodes_explored

# --- Run both algorithms ---
bfs_path, bfs_nodes = bfs_maze(maze, start, goal)
dfs_path, dfs_nodes = dfs_maze(maze, start, goal)
 
print("=== BFS ===")
print("Path:", bfs_path)
print("Path length:", len(bfs_path) if bfs_path else "No path found")
print("Nodes explored:", bfs_nodes)
 
print("\n=== DFS ===")
print("Path:", dfs_path)
print("Path length:", len(dfs_path) if dfs_path else "No path found")
print("Nodes explored:", dfs_nodes)


=== BFS ===
Path: [(0, 0), (1, 0), (2, 0), (2, 1), (2, 2), (3, 2), (4, 2), (4, 3), (4, 4)]
Path length: 9
Nodes explored: 15

=== DFS ===
Path: [(0, 0), (1, 0), (2, 0), (2, 1), (2, 2), (3, 2), (4, 2), (4, 3), (4, 4)]
Path length: 9
Nodes explored: 14


In [49]:
def manhattan(a,b):
    return abs(a[0] - b[0]) + abs(a[1]-b[1])

goal = (4,4)

for cell in [(0, 0), (0, 1), (2, 2), (1, 3), (4, 3), (4, 4)]: 
    print(cell, "-> h =", manhattan(cell, goal)) 


import math

def euclidean(a, b):
    return math.sqrt((a[0] - b[0])**2 + (a[1] - b[1])**2)

goal = (4, 4)

for cell in [(0, 0), (0, 1), (2, 2), (1, 3), (4, 3), (4, 4)]:
    print(cell, "-> h =", round(euclidean(cell, goal), 2))

(0, 0) -> h = 8
(0, 1) -> h = 7
(2, 2) -> h = 4
(1, 3) -> h = 4
(4, 3) -> h = 1
(4, 4) -> h = 0
(0, 0) -> h = 5.66
(0, 1) -> h = 5.0
(2, 2) -> h = 2.83
(1, 3) -> h = 3.16
(4, 3) -> h = 1.0
(4, 4) -> h = 0.0


In [ ]:
import heapq 

graph = { 
"S": {"A": 1, "B": 4}, 
"A": {"S": 1, "C": 1}, 
"B": {"S": 4, "D": 2}, 
"C": {"A": 1, "G": 10}, 
"D": {"B": 2, "G": 3}, 
"G": {"C": 10, "D": 3} 
} 
h = {"S": 4, "A": 3, "B": 4, "C": 2, "D": 3, "G": 0} 
  
def path_cost(graph, path): 
    return sum(graph[path[i]][path[i + 1]] for i in range(len(path) - 1)) 
  
  
def greedy_search(graph, h, start, goal): 
    frontier = [(h[start], [start])]      # (score = h, path) 
    explored = set() 
    nodes_explored = 0 
  
    while frontier: 
        _, path = heapq.heappop(frontier)  # lowest h first 
        current = path[-1] 
  
        if current == goal: 
            return path, nodes_explored 
  
        if current not in explored: 
            explored.add(current) 
            nodes_explored += 1 
            for neighbor in graph[current]: 
                if neighbor not in explored: 
                    heapq.heappush(frontier, (h[neighbor], path + [neighbor])) 
  
    return None, nodes_explored 
  
  
path, nodes = greedy_search(graph, h, "S", "G") 
print("Path:", path) 
print("Cost:", path_cost(graph, path)) 
print("Nodes explored:", nodes) 

Path: ['S', 'A', 'C', 'G']
Cost: 12
Nodes explored: 3


In [50]:
def a_star(graph, h, start, goal): 
    frontier = [(h[start], 0, [start])]     
# (f, g, path) 
    explored = set() 
    nodes_explored = 0 
    while frontier: 
        f, g, path = heapq.heappop(frontier)   # lowest f first 
        current = path[-1] 
        if current == goal: 
            return path, nodes_explored 
        if current not in explored: 
            explored.add(current) 
            nodes_explored += 1 
            for neighbor, step_cost in graph[current].items(): 
                if neighbor not in explored: 
                    new_g = g + step_cost 
                    new_f = new_g + h[neighbor] 
                    heapq.heappush(frontier, (new_f, new_g, path + [neighbor])) 

    return None , nodes_explored

path, nodes = a_star(graph, h, "S", "G") 
print("Path:", path) 
print("Cost:", path_cost(graph, path)) 
print("Nodes explored:", nodes) 

Path: ['S', 'B', 'D', 'G']
Cost: 9
Nodes explored: 5


In [51]:
import heapq 
import numpy as np 
from collections import deque 
  
maze = np.array([ 
    [0, 0, 1, 0, 0], 
    [0, 1, 1, 0, 0], 
    [0, 0, 0, 0, 1], 
    [1, 1, 0, 1, 0], 
    [0, 0, 0, 0, 0] 
]) 
  
start = (0, 0) 
goal = (4, 4) 
  
  
def manhattan(a, b): 
    return abs(a[0] - b[0]) + abs(a[1] - b[1]) 
  
  
def get_neighbors(state, maze): 
    row, col = state 
    candidates = [(row - 1, col), (row + 1, col), (row, col - 1), (row, col + 1)] 
    valid = [] 
    for r, c in candidates: 
        if 0 <= r < maze.shape[0] and 0 <= c < maze.shape[1]: 
            if maze[r, c] == 0: 
                valid.append((r, c)) 
    return valid 
  
  
def bfs_maze(maze, start, goal): 
    frontier = deque([[start]]) 
    explored = set() 
    nodes_explored = 0 
    while frontier: 
        path = frontier.popleft() 
        current = path[-1] 
        if current == goal: 
            return path, nodes_explored 
        if current not in explored: 
            explored.add(current) 
            nodes_explored += 1 
            for neighbor in get_neighbors(current, maze): 
                if neighbor not in explored: 
                    frontier.append(path + [neighbor]) 
    return None, nodes_explored 
  
  
def greedy_maze(maze, start, goal): 
    frontier = [(manhattan(start, goal), [start])] 
    explored = set() 
    nodes_explored = 0 
    while frontier: 
        _, path = heapq.heappop(frontier) 
        current = path[-1] 
        if current == goal: 
            return path, nodes_explored 
        if current not in explored: 
            explored.add(current) 
            nodes_explored += 1 
            for neighbor in get_neighbors(current, maze): 
                if neighbor not in explored: 
                    heapq.heappush(frontier, (manhattan(neighbor, goal), path + [neighbor])) 
    return None, nodes_explored 
  
  
def astar_maze(maze, start, goal): 
    frontier = [(manhattan(start, goal), 0, [start])] 
    explored = set() 
    nodes_explored = 0 
    while frontier: 
        f, g, path = heapq.heappop(frontier) 
        current = path[-1] 
        if current == goal: 
            return path, nodes_explored 
        if current not in explored: 
            explored.add(current) 
            nodes_explored += 1 
            for neighbor in get_neighbors(current, maze): 
                if neighbor not in explored: 
                    new_g = g + 1 
                    new_f = new_g + manhattan(neighbor, goal) 
                    heapq.heappush(frontier, (new_f, new_g, path + [neighbor])) 
    return None, nodes_explored 
  
  
results = { 
    "BFS":    bfs_maze(maze, start, goal), 
    "Greedy": greedy_maze(maze, start, goal), 
    "A*":     astar_maze(maze, start, goal), 
} 
  
for name, (path, nodes) in results.items(): 
    print(f"=== {name} ===") 
    print("Path:", path) 
    print("Path length (steps):", len(path) - 1) 
    print("Nodes explored:", nodes) 
    print() 

=== BFS ===
Path: [(0, 0), (1, 0), (2, 0), (2, 1), (2, 2), (3, 2), (4, 2), (4, 3), (4, 4)]
Path length (steps): 8
Nodes explored: 15

=== Greedy ===
Path: [(0, 0), (1, 0), (2, 0), (2, 1), (2, 2), (3, 2), (4, 2), (4, 3), (4, 4)]
Path length (steps): 8
Nodes explored: 10

=== A* ===
Path: [(0, 0), (1, 0), (2, 0), (2, 1), (2, 2), (3, 2), (4, 2), (4, 3), (4, 4)]
Path length (steps): 8
Nodes explored: 10



In [56]:
def print_maze_with_path(maze, path): 
    display = maze.astype(str) 
    display[display == "0"] = "." 
    display[display == "1"] = "#" 
    for (r, c) in path: 
        display[r, c] = "*" 
    for row in display: 
        print(" ".join(row)) 
print_maze_with_path(maze, results["A*"][0]) 

* . # . .
* # # . .
* * * . #
# # * # .
. . * * *
